# Your First Vector RAG Application: Cat Health Assistant

In this notebook, we will build a dense vector retrieval application using **LangChain v1**, **OpenAI embeddings**, and **Qdrant** as an in-memory vector database.

The goal is to understand the core RAG loop:

1. Load a cat health guideline PDF
2. Split it into smaller chunks
3. Embed those chunks
4. Store the embeddings in Qdrant
5. Retrieve relevant chunks for a question
6. Generate an answer grounded in the retrieved context

> Note: This notebook expects Python 3.12 and uses uv for dependency management.

> Note: This is a vector RAG lesson, not a veterinary care tool. The assistant should answer from the PDF and point users to a veterinarian for diagnosis, treatment, medication, or urgent care decisions.

## Table of Contents

- Task 1: Environment Setup
- Task 2: Embedding Similarity Primer
- Task 3: Documents - Loading the Cat Health Guideline PDF
- Task 4: Chunking the Documents
- Task 5: Embeddings and Qdrant
- Task 6: Retrieval with Scores
- Task 7: Retrieval Augmented Generation
- Activity: Tune Retrieval Quality

## Task 1: Environment Setup

From the `01_Dense_Vector_Retrieval` folder, install dependencies with uv:

```bash
uv sync
```

Then open this notebook in Cursor or VS Code and select the Python/Jupyter environment created by uv.

### Imports

LangChain v1 separates integrations into partner packages. We will use:

- `langchain_community` for PDF loading
- `langchain_text_splitters` for chunking
- `langchain_openai` for chat and embedding models
- `langchain_qdrant` for the Qdrant vector store

In [1]:
from pathlib import Path
from math import sqrt
from getpass import getpass
import os

from langchain_community.document_loaders import PyPDFLoader
from langchain_core.output_parsers import StrOutputParser
from langchain_core.prompts import ChatPromptTemplate
from langchain_openai import ChatOpenAI, OpenAIEmbeddings
from langchain_qdrant import QdrantVectorStore
from langchain_text_splitters import RecursiveCharacterTextSplitter

/tmp/ipykernel_13412/4147450701.py:6: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.document_loaders import PyPDFLoader


### OpenAI API Key

The chat model and embedding model both use OpenAI. If `OPENAI_API_KEY` is not already set in your environment, this cell will ask for it securely.

In [2]:
if not os.environ.get("OPENAI_API_KEY"):
    os.environ["OPENAI_API_KEY"] = getpass("OpenAI API Key: ")

## Task 2: Embedding Similarity Primer

Before we load a full PDF, let's make dense vector retrieval less mysterious.

An embedding model turns text into a list of numbers. Texts with related meaning should land closer together in that vector space.

A common way to score closeness is **cosine similarity**:

```text
cosine_similarity(a, b) = dot_product(a, b) / (length(a) * length(b))
```

The intuition: if two vectors point in a similar direction, their cosine similarity is higher. Vector databases like Qdrant use this same idea, but at a much larger scale.

In [3]:
embedding_model = "text-embedding-3-small"
embeddings = OpenAIEmbeddings(model=embedding_model)

example_texts = [
    "king",
    "queen",
    "banana",
    "cat",
    "veterinarian",
    "cat health guidelines",
]

example_vectors = dict(zip(example_texts, embeddings.embed_documents(example_texts)))


def cosine_similarity(vector_a: list[float], vector_b: list[float]) -> float:
    dot_product = sum(a * b for a, b in zip(vector_a, vector_b))
    length_a = sqrt(sum(a * a for a in vector_a))
    length_b = sqrt(sum(b * b for b in vector_b))
    return dot_product / (length_a * length_b)


comparison_pairs = [
    ("king", "queen"),
    ("king", "banana"),
    ("cat", "veterinarian"),
    ("cat", "cat health guidelines"),
]

for left, right in comparison_pairs:
    score = cosine_similarity(example_vectors[left], example_vectors[right])
    print(f"{left:>22} <> {right:<22} score={score:.3f}")

                  king <> queen                  score=0.591
                  king <> banana                 score=0.310
                   cat <> veterinarian           score=0.356
                   cat <> cat health guidelines  score=0.496


A few important notes:

- The score is useful for ranking, not as an absolute truth about meaning.
- Different embedding models can produce different scores.
- In RAG, we embed each document chunk once, then embed the user's query and search for the nearest chunk vectors.

That is the retrieval part of RAG.

## Task 3: Documents

LangChain represents loaded text as `Document` objects. A `Document` has:

- `page_content`: the text
- `metadata`: information such as source file and page number

We will load one `Document` per PDF page, then split those pages into smaller chunks.

### Course PDF

This notebook uses the bundled cat health guideline PDF at:

```text
01_Dense_Vector_Retrieval/data/cat_health_guidelines.pdf
```

The next cell checks that the course material is present before we start loading pages.

In [4]:
pdf_path = Path("data/cat_health_guidelines.pdf")

if not pdf_path.exists():
    raise FileNotFoundError(
        f"Expected the cat health guideline PDF at: {pdf_path.resolve()}\n"
        "The bundled course PDF is missing from this copy of the materials."
    )

### Load the PDF

`PyPDFLoader` extracts text from text-based PDFs. If the PDF is scanned images, this loader may return little or no text, and OCR would be needed.

In [5]:
loader = PyPDFLoader(str(pdf_path))
pages = loader.load()

for page in pages:
    page.metadata["source"] = pdf_path.name
    page.metadata["document_type"] = "cat_health_guideline"

pages = [page for page in pages if page.page_content.strip()]

if not pages:
    raise ValueError(
        "The PDF loaded, but no extractable text was found. "
        "This usually means the PDF is scanned and needs OCR first."
    )

print(f"Loaded {len(pages)} text-containing PDF pages.")

Loaded 22 text-containing PDF pages.


In [6]:
print(pages[0].page_content[:750])
print("\nMetadata:", pages[0].metadata)

VETERINARY PRACTICE GUIDELINES
2021 AAHA/AAFP Feline Life Stage Guidelines*
Jessica Quimby, DVM, PhD, DACVIM y, Shannon Gowland, DVM, DABVP y, Hazel C. Carney, DVM, MS, DABVP,
Theresa DePorter, DVM, MRCVS, DACVB, DECAWBM, Paula Plummer, LVT, VTS (ECC, SAIM), Jodi Westropp,
DVM, PhD, DACVIM
ABSTRACT
The guidelines, authored by a Task Force ofexperts in feline clinical medicine, are an update and extension of the AAFP–AAHA
Feline Life Stage Guidelines published in 2010. The guidelines are published simultaneously in theJournal of Feline Medicine and
Surgery(volume 23, issue 3, pages 211–233, DOI: 10.1177/1098612X21993657) and theJournal of the American Animal Hospital
Association(volume 57, issue 2, pages 51–72, DOI: 10.5326/JAAHA-MS-7189). A

Metadata: {'producer': 'Acrobat Distiller 10.0.0 (Windows)', 'creator': 'Adobe InDesign CS6 (Windows)', 'creationdate': '2021-02-02T08:52:15-05:00', 'author': '7123', 'moddate': '2021-02-02T07:53:51-07:00', 'title': 'djs_jaaha_56_5_COVER.indd', 'so

#### ❓Question #1

Why is metadata important for a RAG application?

##### ✅ Answer:

_Having metadata in RAG can benefit us in several ways:_
* _Allows us to __pre-filter__ documents or the vector search space before semantic search, which can be costly and noisy and improves retrieval relevancy. It can also serve as a layer of access control._
* _While chunking may lose general context tied to the whole document, metadata __can carry that context forward__ to each chunk._
* _Enables __hybrid search__, combining exact matching with semantic search when both precision and recall matter._
* _Plays a critical role in __evaluation__ for tracing and measuring the retrieval module itself, metadata is a key source for computing faithfulness and context precision_

## Task 4: Chunking the Documents

A full PDF page can be too large or too mixed-topic for high-quality retrieval. We split pages into overlapping chunks so each chunk has enough local context but is still focused.

Here we will start with chunks of 1,000 characters and 200 characters of overlap. The chunk size controls how much text each vector represents; the overlap keeps nearby context from being lost at chunk boundaries.

`RecursiveCharacterTextSplitter` tries to split on natural boundaries first, such as paragraphs and line breaks, before falling back to smaller separators.

In [7]:
chunk_size = 1000
chunk_overlap = 200

text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=chunk_size,
    chunk_overlap=chunk_overlap,
    add_start_index=True,
)

splits = text_splitter.split_documents(pages)

print(f"Split {len(pages)} pages into {len(splits)} chunks.")
print(f"Chunk size: {chunk_size} characters")
print(f"Chunk overlap: {chunk_overlap} characters")

Split 22 pages into 135 chunks.
Chunk size: 1000 characters
Chunk overlap: 200 characters


In [8]:
sample_chunk = splits[0]
print(sample_chunk.page_content[:750])
print("\nMetadata:", sample_chunk.metadata)

VETERINARY PRACTICE GUIDELINES
2021 AAHA/AAFP Feline Life Stage Guidelines*
Jessica Quimby, DVM, PhD, DACVIM y, Shannon Gowland, DVM, DABVP y, Hazel C. Carney, DVM, MS, DABVP,
Theresa DePorter, DVM, MRCVS, DACVB, DECAWBM, Paula Plummer, LVT, VTS (ECC, SAIM), Jodi Westropp,
DVM, PhD, DACVIM
ABSTRACT
The guidelines, authored by a Task Force ofexperts in feline clinical medicine, are an update and extension of the AAFP–AAHA
Feline Life Stage Guidelines published in 2010. The guidelines are published simultaneously in theJournal of Feline Medicine and
Surgery(volume 23, issue 3, pages 211–233, DOI: 10.1177/1098612X21993657) and theJournal of the American Animal Hospital
Association(volume 57, issue 2, pages 51–72, DOI: 10.5326/JAAHA-MS-7189). A

Metadata: {'producer': 'Acrobat Distiller 10.0.0 (Windows)', 'creator': 'Adobe InDesign CS6 (Windows)', 'creationdate': '2021-02-02T08:52:15-05:00', 'author': '7123', 'moddate': '2021-02-02T07:53:51-07:00', 'title': 'djs_jaaha_56_5_COVER.indd', 'so

#### ❓Question #2

What tradeoff do we make when choosing chunk size and chunk overlap?

##### ✅ Answer:

_Both too-small and too-large chunk sizes introduce their own tradeoffs:_

* _Smaller chunks produce more precise retrieval but may lose surrounding context needed to fully answer a query. They also increase storage and retrieval overhead._
* _Larger chunks carry more context per chunk but may introduce noise or irrelevant content that dilutes the embedding._

_For chunk overlap, choosing no overlap risks losing continuity at chunk boundaries, while high overlap introduces redundancy and increases retrieval cost._

_There is no universally optimal strategy and each document type and use case needs its own chunking decision, strategies, tuned and validated against retrieval metrics._

Use the [Chunk Visualizer](https://chunkviz.up.railway.app/) to experiment with different chunk sizes and overlaps and see how the text boundaries change.

## Task 5: Embeddings and Qdrant

Now we apply the same embedding idea to every chunk from the PDF. Qdrant stores those vectors and lets us search for chunks that are close to a query in embedding space.

We already created an OpenAI embedding model in the primer above. The Qdrant collection name is just a label for the set of vectors we are creating.

For this notebook, Qdrant runs in memory with `location=":memory:"`. That means no Docker, no Qdrant Cloud account, and no persistence after the notebook kernel stops.

In [9]:
collection_name = "cat_health_guidelines"

vector_store = QdrantVectorStore.from_documents(
    documents=splits,
    embedding=embeddings,
    location=":memory:",
    collection_name=collection_name,
    force_recreate=True,
)

print(f"Embedded chunks with: {embedding_model}")
print(f"Built in-memory Qdrant collection: {collection_name}")

Embedded chunks with: text-embedding-3-small
Built in-memory Qdrant collection: cat_health_guidelines


## Task 6: Retrieval with Scores

Before we generate answers, we should inspect retrieval directly. If retrieval returns poor context, the final answer will usually be poor too.

The value `k` controls how many chunks the retriever returns. A larger `k` gives the model more context, but it can also add noise. We will start with `k = 4` and tune it later.

Qdrant can return both the matching `Document` and a similarity score. This is the same ranking idea we saw with `king`, `queen`, and `cat`, now applied to PDF chunks.

In [10]:
def display_retrieval_results(query: str, k: int) -> list[tuple]:
    """Retrieve chunks and print a compact view of the results."""
    results = vector_store.similarity_search_with_score(query, k=k)

    for index, (doc, score) in enumerate(results, start=1):
        page = doc.metadata.get("page")
        page_display = page + 1 if isinstance(page, int) else "unknown"
        start_index = doc.metadata.get("start_index", "unknown")
        preview = doc.page_content[:350].replace("\n", " ")

        print(f"Source {index} | score={score:.3f} | page={page_display} | start_index={start_index}")
        print(preview)
        print("-" * 80)

    return results

In [11]:
retrieval_k = 4
retrieval_query = "What signs suggest that a cat should be seen by a veterinarian?"
retrieved_results = display_retrieval_results(retrieval_query, k=retrieval_k)

Source 1 | score=0.584 | page=8 | start_index=0
Detecting signs of pain or anxiety and evaluation of quality of life are most commonly of concern in the mature adult or senior cat but may be relevant at any life stage. During the physical examination, particular focus is on pain assessment and abdominal and thyroid palpation. A detailed mus- culoskeletal examination to detect signs of osteoarthr
--------------------------------------------------------------------------------
Source 2 | score=0.571 | page=7 | start_index=2384
Asking speci ﬁc questions concerning whether vomiting, vom- iting hairballs, or diarrhea is occurring, and the frequency of each, is recommended as some clients may consider vomiting or vomiting hairballs to be normal for their cat. Additionally, discuss the im- portance of monitoring weight, and ask about any chronic enter- opathy or gastrointesti
--------------------------------------------------------------------------------
Source 3 | score=0.565 | page=7 | sta

#### ❓Question #3

What does a similarity score help you understand, and what does it not prove by itself?

##### ✅ Answer:

_Embedding vectors are placed in a high-dimensional vector space. The similarity score reflects how geometrically close two vectors are in that space. The higher the score, the more likely the two words, sentences, or documents share similar meaning in natural language. However, this is not guaranteed, because the features captured by an embedding model may not align with how humans interpret language and are not always easily interpretable._

_Additionally, similarity is embedding-model-dependent; So, the same text can produce different scores across different models._

_Another subtle issue: antonyms and opposing concepts often appear close in vector space because they share similar context windows during training. This makes their relationship hard to infer from vectors alone._

_Finally, since we are computing similarity between a query and chunks, a high score does not necessarily mean the chunk is useful or answers the query. It only reflects topical closeness, not answer quality or factual correctness._


## Task 7: Retrieval Augmented Generation

Now we combine retrieval with generation. We will use a two-step RAG pattern:

1. Retrieve relevant chunks from Qdrant
2. Put those chunks into the prompt and ask the model to answer from the context

This is intentionally simpler than an agent. We always retrieve before answering, which makes the vector retrieval mechanics easy to inspect.

For generation, we will use `gpt-5.4-mini`.

In [12]:
chat_model = "gpt-5.4-mini"
llm = ChatOpenAI(model=chat_model)

RAG_SYSTEM_PROMPT = """You are a cat health guideline assistant in a vector RAG lesson.

Use only the provided context to answer the user's question.
If the context does not contain enough information, say: "I don't have enough information in the provided cat health guideline PDF to answer that."

Cite the retrieved sources inline using labels like [Source 1] or [Source 2].
Do not diagnose, prescribe medication, or replace a veterinarian.
For diagnosis, treatment decisions, medication questions, or urgent symptoms, recommend contacting a veterinarian.
Keep the answer concise and practical."""

RAG_USER_PROMPT = """Context:
{context}

Question: {question}

Answer from the context above."""

rag_prompt = ChatPromptTemplate.from_messages(
    [
        ("system", RAG_SYSTEM_PROMPT),
        ("human", RAG_USER_PROMPT),
    ]
)

rag_chain = rag_prompt | llm | StrOutputParser()

In [13]:
def format_context(scored_docs: list[tuple]) -> str:
    """Convert retrieved documents into a source-labeled context string."""
    formatted_chunks = []

    for index, (doc, score) in enumerate(scored_docs, start=1):
        page = doc.metadata.get("page")
        page_display = page + 1 if isinstance(page, int) else "unknown"
        source = doc.metadata.get("source", "unknown source")

        formatted_chunks.append(
            f"[Source {index}] {source}, page {page_display}, score {score:.3f}\n"
            f"{doc.page_content}"
        )

    return "\n\n".join(formatted_chunks)


def answer_question(question: str, k: int) -> dict:
    """Run retrieve-then-generate and return the answer plus source metadata."""
    scored_docs = vector_store.similarity_search_with_score(question, k=k)
    context = format_context(scored_docs)
    answer = rag_chain.invoke({"context": context, "question": question})

    sources = []
    for index, (doc, score) in enumerate(scored_docs, start=1):
        page = doc.metadata.get("page")
        sources.append(
            {
                "source_label": f"Source {index}",
                "file": doc.metadata.get("source"),
                "page": page + 1 if isinstance(page, int) else None,
                "start_index": doc.metadata.get("start_index"),
                "score": score,
            }
        )

    return {
        "question": question,
        "answer": answer,
        "sources": sources,
        "context": scored_docs,
    }

Before calling the model, inspect the formatted context. This is the exact text that will be inserted into the RAG prompt.

In [14]:
example_context = format_context(retrieved_results[:2])
print(example_context[:2000])

[Source 1] cat_health_guidelines.pdf, page 8, score 0.584
Detecting signs of pain or anxiety and evaluation of quality of life
are most commonly of concern in the mature adult or senior cat but
may be relevant at any life stage.
During the physical examination, particular focus is on pain
assessment and abdominal and thyroid palpation. A detailed mus-
culoskeletal examination to detect signs of osteoarthritis is critical as
this condition is one of the most signi ﬁcant and underdiagnosed
diseases in cats.
23,28 A fundic examination is key to detecting signs of
ophthalmic disease or hypertension. 29 Practices should employ a
validated pain assessment scale or tool to diagnose, monitor, and
assist in the evaluation of patients for subtle signs of pain.
30
Changes in grooming habits, particularly increased grooming,
may signal a dermatologic issue such as atopy, food allergy, an
immune-mediated skin condition, infectious or parasitic disease,
endocrine condition, or paraneoplastic syndrom

In [15]:
answer_k = 4

result = answer_question(
    "What are signs that my cat may need veterinary attention?",
    k=answer_k,
)

print(result["answer"])
print("\nSources:")
for source in result["sources"]:
    print(source)

Signs that your cat may need veterinary attention can include:

- Pain or anxiety signs, especially in mature adult or senior cats, though they can happen at any age [Source 1]
- Changes in grooming, such as increased grooming or reduced grooming [Source 1][Source 2]
- Reduced grooming, which may also suggest illness, bladder pain, joint pain, or reduced mobility [Source 2]
- Vomiting, vomiting hairballs, diarrhea, or other chronic GI signs [Source 3]
- Changes in appetite, weight, thirst/urination, or normal habits/activity [Source 3]
- Increased nocturnal activity or vocalization [Source 3]
- Signs of fear, stress, or distress such as cowering, crouching, hiding, freezing, flat or rotated ears, dilated pupils, tense body posture, or hissing/yowling/growling/screaming [Source 4]
- House-soiling or aggression can also be concerning behavior changes [Source 2]

If you’re seeing any of these, especially if they are new, persistent, or worsening, contact a veterinarian for an evaluation.


### Vibe Check Queries

Run a few questions that should be answerable from a cat health guideline PDF. Then run one question that may not be answerable and confirm the assistant says it does not have enough information.

In [ ]:
vibe_check_questions = [
    "What preventive care is recommended for cats?",
    "What symptoms should make me call a veterinarian?",
    "What should I know about feeding a healthy adult cat?",
    "Can my cat help me file my taxes?",
]

for question in vibe_check_questions:
    print("\nQuestion:", question)
    result = answer_question(question, k=answer_k)
    print('\nRetrieved sources:')
    for index, (source, score) in enumerate(result['context']):
        print(f'\t- Source {index} | score={score:.3f} | content={source.page_content.replace("\n", " ")[:500]}...')
    print("\nAnswer:")
    print(result["answer"])
    print("=" * 100)


Question: What preventive care is recommended for cats?

Retrieved sources:
	- Source 0 | score=0.657 | content=17,120,121 This approach will eliminate existing infections, as well as decrease the risk of further infestation and subsequent associated clinical problems. Canine and feline housemates may be at risk of transmission of infectious parasites including roundworm and ﬂeas and therefore should be treated in synchronicity with newly acquired kittens or cats. Pre- venting cats ’ access to gardens and children ’s play sand areas will, combined with parasite prophylaxis, decrease environmental con- tam...
	- Source 1 | score=0.648 | content=alized risk assessment, preventive healthcare strategies, and treat- ment pathways that evolve as the cat matures. An evidence-guided framework for managing a cat ’s healthcare throughout its lifetime has never been more important in feline practice than it is now. Cats are the most popular pet in the United States. 1 A great anomaly in feline p

#### ❓Question #4

For the vibe check queries above, did the retrieved context seem relevant before generation? Why or why not?

##### ✅ Answer:

_I've changed the code in the cell a bit to see what the actual retrieved chunks are for each question. It seems for each question, the retriever tried to find the most similar chunks existing across all chunks. In detail:_
- __What preventive care is recommended for cats?__ _Scores of retrieved chunks are relatively high, so they are supposed to have a similar context to the question and we expect to find the answer in these chunks. Reviewing them in detail shows that while we get relevant answers in the first and last chunks, the second one is semi-relevant and the third one is bibliography with no actionable preventive care content! It's a common issue because bibliography chunks likely have high semantic overlap but carry zero answerable content._
- __What symptoms should make me call a veterinarian?__ _The first two chunks are completely relevant and cover the whole list of symptoms, the fourth chunk is partially relevant, but the third one is off-topic._
- __What should I know about feeding a healthy adult cat?__ _Scores show high semantic relevancy between the question and chunks, and reviewing the chunks, it seems the best retrieved set among these questions because they are highly relevant and the order is reasonable; the last one is not completely relevant but also not irrelevant._
- __Can my cat help me file my taxes?__ _Reviewing the retrieved chunks, scores are lower than the other questions, signaling weak semantic overlap. Although they might be the most relevant chunks in the whole set, there is no meaningful similarity between the chunks and the question at all. The retriever is forced to return a fixed top-k chunks. So both the retriever and LLM did their job well. This was out-of-domain, and we don't expect cats to do our taxes, but we hope AI agents do :))_

## 🏗️ Activity: Tune Retrieval Quality

Improve retrieval quality by changing one or more of these values:

- The chunk size
- The chunk overlap
- The retrieval `k`
- The wording of the retrieval query

Suggested workflow:

1. Pick one test question.
2. Inspect the retrieved chunks and scores.
3. Change one retrieval setting.
4. Rebuild the splitter and vector store.
5. Compare whether the retrieved chunks became more relevant.

When you are done, write down what changed and whether the final answer improved.

In [16]:
# Initial experiment
test_query = "What preventive care is recommended for cats?"

print("Initial experiment settings with chunk-size=1000 and chunk-overlap=200")
results=display_retrieval_results(test_query, k=retrieval_k)
print(results)


Initial experiment settings with chunk-size=1000 and chunk-overlap=200
Source 1 | score=0.657 | page=15 | start_index=1549
17,120,121 This approach will eliminate existing infections, as well as decrease the risk of further infestation and subsequent associated clinical problems. Canine and feline housemates may be at risk of transmission of infectious parasites including roundworm and ﬂeas and therefore should be treated in synchronicity with newly acquired kittens or
--------------------------------------------------------------------------------
Source 2 | score=0.648 | page=2 | start_index=821
alized risk assessment, preventive healthcare strategies, and treat- ment pathways that evolve as the cat matures. An evidence-guided framework for managing a cat ’s healthcare throughout its lifetime has never been more important in feline practice than it is now. Cats are the most popular pet in the United States. 1 A great anomaly in feline prac
--------------------------------------------

In [ ]:
# Activity workspace

def single_experiment_on_retrieval(chunk_size, chunk_overlap, top_k=4, test_query=test_query):

    experiment_name = f'{chunk_size}-char_{chunk_overlap}-overlap_top-{top_k}'

    splitter = RecursiveCharacterTextSplitter(
        chunk_size=chunk_size,
        chunk_overlap=chunk_overlap,
        add_start_index=True,
    )

    splits = splitter.split_documents(pages)
    
    print(f"The document splitted to {len(splits)} chunks with chunk-size={chunk_size} and chunk-overlap={chunk_overlap}")

    vector_store = QdrantVectorStore.from_documents(
        documents=splits,
        embedding=embeddings,
        location=":memory:",
        collection_name=f"cat_health_tuning_{experiment_name}",
        force_recreate=True,
    )

    tuned_results = vector_store.similarity_search_with_score(test_query, k=top_k)

    for i, (doc, score) in enumerate(tuned_results, start=1):
        page = doc.metadata.get("page")
        page_display = page + 1 if isinstance(page, int) else "unknown"
        preview = doc.page_content.replace("\n", " ")
        print(f"Source {i} | score={score:.3f} | page={page_display}")
        print(preview)
        print("-" * 80)

    print('\n', '=' * 100, '\n\n')


# Retrieval tuning
experiments = [
    {"chunk_size": 1000, "chunk_overlap": 200, "top_k":4},  # baseline
    {"chunk_size": 1000, "chunk_overlap": 200, "top_k":5},  # top-k change on baseline
    {"chunk_size": 1000, "chunk_overlap": 100, "top_k":5},  # overlap change on baseline
    {"chunk_size": 500,  "chunk_overlap": 100, "top_k":5},  # small chunk, proportional overlap
    {"chunk_size": 500,  "chunk_overlap": 200, "top_k":5},  # small chunk, high overlap ratio
    {"chunk_size": 1500, "chunk_overlap": 200, "top_k":5},  # large chunk, same overlap
    {"chunk_size": 1500, "chunk_overlap": 300, "top_k":5},  # large chunk, proportional overlap
]


for exp in experiments:
    single_experiment_on_retrieval(**exp)

The document splitted to 135 chunks with chunk-size=1000 and chunk-overlap=200
Source 1 | score=0.657 | page=15
17,120,121 This approach will eliminate existing infections, as well as decrease the risk of further infestation and subsequent associated clinical problems. Canine and feline housemates may be at risk of transmission of infectious parasites including roundworm and ﬂeas and therefore should be treated in synchronicity with newly acquired kittens or cats. Pre- venting cats ’ access to gardens and children ’s play sand areas will, combined with parasite prophylaxis, decrease environmental con- tamination with infectious and zoonotic agents such as hookworms and Toxoplasma gondii. 33 Routine, regular use of broad-spectrum products is likely to be beneﬁcial for the majority of pet cats, regardless of lifestyle. Certain outdoor lifestyles, geographic location, and whether a cat spends time away from the home (travel, boarding facilities, groomer, etc.) may increase the existing ri

### 🏗️ Activity Notes

- __Setting changed:__ _Designed 6 experiments varying chunk-size, chunk-overlap, and top-k_
- __Before:__ _chunk-size=1000, chunk-overlap=200, k=4_
- __After:__ _Best result at chunk-size=1500, overlap=200, k=5_
- __Did retrieval improve? Why or why not?__ _Across all trials, one setting produced noticeably better results while others degraded quality, so there is a relative improvement, but it's not uniform. Here's what each experiment revealed:_

  - __Experiment 1 & 2__ (chunk-size=1000, overlap=200) _Identical results across both runs, confirming deterministic retrieval. Two high-relevance chunks (parasite control, exam frequency), two medium (lifestyle framing), and one pure bibliography entry ranking 3rd which is the main weakness. The bibliography gets inflated by keyword overlap with terms like "vaccination" and "guidelines" in reference titles._

  - __Experiment 3__ _(chunk-size=500, overlap=100) _The worst setting. The top result is a single CDC URL from a bibliography, and the remaining four are partial paragraph fragments covering the right topics but without depth. Small chunks can't differentiate reference noise from real content, and they break up logically connected sections across multiple incomplete hits._

  - **Experiment 4** (chunk-size=500, overlap=200) _No bibliography contamination, but all five results are medium relevance and scores are the lowest overall. The higher overlap helps slightly, but 333 chunks is significant index overhead for results that are shallower than larger-chunk experiments._

  - **Experiment 5** (chunk-size=1500, overlap=200) _The best setting. Four of five chunks are high relevance, collectively covering parasite control, vaccination, oral health, zoonosis prevention, and life-stage diagnostics so they form nearly a complete answer. Larger chunks keep full sections intact rather than fragments, and no bibliography noise appears._

  - **Experiment 6** (chunk-size=1500, overlap=300) _Essentially tied with Experiment 5. The extra overlap causes minor cross-boundary bleed but doesn't meaningfully change the results: still 4 high, 1 medium, zero off-topic._

  _Chunk size matters far more than overlap. Larger chunks (1500 tokens) keep complete topical sections together and produce cleaner results, while smaller chunks (500 tokens) fragment content and let bibliography entries compete with real content. The best setting found is chunk-size=1500, overlap=200 broadest coverage, no noise, and manageable index size._

  _This conclusion is based on a single test question and cannot be generalized. For reliable evaluation, a comprehensive approach is needed; using frameworks like RAGAS and metrics such as context relevancy, faithfulness, and answer correctness across a diverse question set._


